# Semantic vs Phoneme — DySO decomposition

Goal: isolate word-semantic vs word-phoneme geometry in neural space using DySO + ridge regression.

Pipeline per patient × time bin:
1. Per-trial GloVe (semantic, S) and panphon (phoneme, P) embeddings.
2. PCA each to a common dim; DySO on the pair returns three orthonormal embedding-side bases (U_S_emb, U_P_emb, U_sh_emb).
3. Build per-trial targets in each subspace, ridge-regress neural → target, QR-orthonormalize → neural axes (U_sem_neural, U_phon_neural, U_shared_neural).
4. Project trials onto each neural-axis basis for visualization. K-fold CV R² gives the headline metric.
5. (Optional) permutation null for significance.

Companion script: `main/tests/semantic_phoneme_dyso.py`.

In [ ]:
%matplotlib inline
import os, sys, gc
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd().parents[1] if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))
sys.path.insert(0, str(PROJECT_ROOT / 'main'))

from main.tests import semantic_phoneme_dyso as spd
print('module:', spd.__file__)

## 1. Smoke test: AA, single bin

In [ ]:
PATIENT, BIN = 'AA', 20   # bin 20 = picture-naming peak for AA loose category
RUN = spd.DEFAULT_PIC_RUN

d = spd.load_results_pkl(RUN, PATIENT)
reg = d['regressors']['GloVe']
meta = spd.get_trial_metadata(reg)
words = meta['word'].values

S = reg.y                                  # GloVe per trial
P_full = spd.load_panphon_embeddings(words)
keep = ~np.any(np.isnan(P_full), axis=1)
S = S[keep]; P = P_full[keep]; meta = meta.loc[keep].reset_index(drop=True)
X = reg.X_to_use[BIN][keep]
print('n_trials:', X.shape[0], '  X:', X.shape, '  S:', S.shape, '  P:', P.shape)

## 2. Full decomposition at the chosen bin

In [ ]:
dec = spd.decompose_neural_at_bin(X, S, P, d_common=12, ridge_alpha=1.5)
print('embedding-side DySO variance explained:')
for k, v in dec['var_explained'].items(): print(f'  {k}: {v}')
print('\nneural-axis dims  sem:', dec['U_sem_neural'].shape,
      'phon:', dec['U_phon_neural'].shape,
      'shared:', dec['U_shared_neural'].shape)
print('trial-projection shapes  T_sem:', dec['T_sem'].shape,
      'T_phon:', dec['T_phon'].shape, 'T_shared:', dec['T_shared'].shape)

## 3. Cross-validated R² (the honest number)

In [ ]:
cv = spd.cross_validated_r2(X, S, P, d_common=12, ridge_alpha=1.5, n_splits=5)
import pandas as pd
print(pd.Series(cv).to_string())

# Interpretation:
# - R2_S_on_sem  : how well U_sem captures semantic info (should be ↑)
# - R2_P_on_phon : how well U_phon captures phonological info (should be ↑)
# - R2_S_on_phon : leakage of semantic into phoneme subspace (should be ↓ or ~0)
# - R2_P_on_sem  : leakage of phoneme into semantic subspace (should be ↓ or ~0)
# A clean decomposition shows the diagonal high and off-diagonal near zero.

## 4. 3D scatter — trials in orthogonal neural space (matplotlib)

In [ ]:
spd.plot_3d_scatter(dec['T_sem'], dec['T_phon'], dec['T_shared'],
                     meta, Path('scatter_3d_inline.png'))
from IPython.display import Image, display
display(Image(filename='scatter_3d_inline.png'))

## 5. Interactive 3D scatter (Plotly — rotate with the mouse)

In [ ]:
import plotly.graph_objects as go

x = dec['T_sem'][:, 0] if dec['T_sem'].shape[1] else np.zeros(len(meta))
y = dec['T_phon'][:, 0] if dec['T_phon'].shape[1] else np.zeros(len(meta))
z = dec['T_shared'][:, 0] if dec['T_shared'].shape[1] else np.zeros(len(meta))

fig = go.Figure()
for cat in sorted(meta['category'].unique()):
    mask = (meta['category'].values == cat)
    fig.add_trace(go.Scatter3d(
        x=x[mask], y=y[mask], z=z[mask],
        mode='markers+text',
        marker=dict(size=5, color=spd.CATEGORY_COLORS.get(cat, '#999')),
        text=[f'{w}({c})' for w, c in zip(meta.loc[mask,'word'].values,
                                            meta.loc[mask,'cluster'].values)],
        textposition='top center', textfont=dict(size=8),
        name=cat,
    ))
fig.update_layout(
    title='Trials in orthogonal neural space (color=category, label=word(cluster))',
    scene=dict(xaxis_title='U_sem PC1', yaxis_title='U_phon PC1', zaxis_title='U_shared PC1'),
    width=800, height=600, margin=dict(l=0, r=0, t=40, b=0),
)
fig.show()

## 6. Time-resolved sweep (this bin's neighborhood)

Run a range of bins and see when the semantic axis vs phoneme axis carries information.

**Hypothesis**: in motor cortex, R²(S | U_sem) should peak earlier than R²(P | U_phon) — semantics arrives first, phonological output later.

In [ ]:
BIN_LO, BIN_HI = max(reg.n_bins_history, 10), min(reg.n_bins, 60)
print(f'Sweeping bins {BIN_LO}..{BIN_HI-1} for patient {PATIENT}')
rows = []
for b in range(BIN_LO, BIN_HI):
    Xb = reg.X_to_use[b][keep]
    cv = spd.cross_validated_r2(Xb, S, P, d_common=12, ridge_alpha=1.5, n_splits=5)
    rows.append({'bin_index': b, **cv})
df = pd.DataFrame(rows)
df.head()

In [ ]:
spd.plot_traces(df, Path('dyso_traces_inline.png'),
                bin_size_ms=int(d.get('bin_size_ms', 100)))
from IPython.display import Image, display
display(Image(filename='dyso_traces_inline.png'))

## 7. Word-trajectory quiver in (U_sem PC1, U_phon PC1) plane

For each unique word, mean projection across trials at each bin → trajectory from early to late.

In [ ]:
per_bin_proj = {}
for b in range(BIN_LO, BIN_HI):
    Xb = reg.X_to_use[b][keep]
    dec_b = spd.decompose_neural_at_bin(Xb, S, P, d_common=12, ridge_alpha=1.5)
    per_bin_proj[b] = {'T_sem': dec_b['T_sem'],
                       'T_phon': dec_b['T_phon'],
                       'T_shared': dec_b['T_shared']}
spd.plot_word_trajectory_quiver(per_bin_proj, meta, (BIN_LO, BIN_HI-1),
                                 Path('quiver_inline.png'),
                                 bin_size_ms=int(d.get('bin_size_ms', 100)))
display(Image(filename='quiver_inline.png'))

## 8. Run for all 6 patients

Or use the CLI:
```bash
python -m main.tests.semantic_phoneme_dyso
python -m main.tests.semantic_phoneme_dyso --n-perm 200    # with null distribution
python -m main.tests.semantic_phoneme_dyso --task auditory_naming
```

Then aggregate with the HTML report:
```bash
python -m main.report.semantic_phoneme_dyso_report
```